# Text mining using LLM

In this exercise of the block course, we will extract information about damages (ie. impacts) of natural hazards, such as floods or heatwaves, on critical infrastructure. 
The far-reaching societal and economical impacts of the low water levels in many rivers is an example. The limited transport of goods and products via inland waterways highlights what kind of impacts we can experience in our daily life, but also on the economy, when some of the critical infrastructure is not operational anymore. Besides economical impacts, we also experience cascading impacts in Europe caused by the low water levels in rivers - for example: some reactors of a huge nuclear power plant had to be turned down in Romania (-> with further limitations in the energy supply). The reactors got too less and (too warm) cooling water from the nearby Danube river.

In detail, we focus on the extraction of direct impacts from natural hazards, e.g. disruptions and closures in the transport network, power blackouts, partly operational wastewater treatment plants. 

## Goal of the exercise: 
We build two LLM agents which are interacting with each other in a very simplified way (just one direction) - feel free to adapt the exercise code with your ideas - e.g. interaction loop of LLM agents, more tool calls etc.

#### Approach Idea of this Exercise:
1. One LLM (a GPT model) has to extract information about damaged critical infrastructure from a single scientific publication (but feel free to load more documents). It convert the unstructured textual information into structured data. 
2. Then a second LLM (a Llama model) is called and tries to verify if the response of the first LLm for each case of damaged  infrastructure is correct or not. It flags incorrect responses with a <False>

> NOTE: We dont conduct any model training of the LLM - for the respective task it would be too time-consuming and would quite a lot of computational resources 

## Load packages and your secret key

In [ ]:
import os
from glob import glob
from pathlib import Path
import ast
import re
from typing import Annotated, List, Literal
from annotated_types import Len

import pandas as pd
import spacy
from langchain.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, END
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel


test_mode = True

#  automatic linebreaks and multi-line cells.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.colheader_justify", "left")

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# my_secrete_key = os.getenv("NVIDIA_TOKEN")


## Alternative : execute the following command directly in your temrinal as environment variable
# export NVIDIA_API_KEY="nvapi-..."
## For Windows PowerShell
# $env:NVIDIA_API_KEY = "nvapi-...".

#### Download the spacy language transformer model 

We need this model to detect "keywords" in the text that refer to critical infrastructure assets.
Actually, this model is conducting a traditional Natural Language Processing task called **Named Entity Recognition (NER)** . 
An entity could be for example, any kind of numerical value, a person name, or a location. SpaCy language models are trained ML models (or the very large ones are transformer models as in our exercise). They were trained for identifying different these different entities from text, but they can be also applied with an user-defined entity class:
In our exercise the spaCy model needs to detect two entity classes :
1. A pre-existing entity class called "FAC" (it refers to Words like "Frankfurt airport", "London Heathrow", or highway names like "A 3" or "B150")  
2. A customised entity class (see, `ner_patterns.jsonl/pattern`) specifically created for detecting Infrastructure "words" from text - I simply used Regular expression to find words like "bridge(s)", "waterway", "offshore park", etc. - all of them describe critical infrastructure objects


**Try out to execute following code line:**

In [ ]:
nlp = spacy.load(
    "./spacy_model_pipeline"
)  # try to load the spacy model in the "spacy_model_pipeline" folder, the model is needed for entity recognition

**If you get an error then download spacy transformer model**
NOTE:\ 
Simply execute the two code lines below just once - they download the spacy model (for the english language). The model is needed for entity recognition.\
If needed: replace the `"./spacy_model_pipeline"` everywhere in the notebook with `en_core_web_trf`


In [ ]:
# spacy.cli.download('en_core_web_trf')  # <- you can also try out smaller models (see: https://spacy.io/models/en), for example, the "en_core_web_sm" model (a small ML model for the english language)
# nlp = spacy.load('en_core_web_trf')

## Load the Data


### Task
* Load the text-sources on which we aim to conduct the extraction

**Good practice:**
Place your data outside of the project folder (ie. the root directory of Git - its where `.git` is located ). In other words, your `data` should always be located in a parent directory of `.git`.


In [ ]:
#  input data dir
# NOTE : setting the path to the data folder is not the best approach but (for now) very convenient ;)
# Better approach would be, for example, a config file (similar as done in the Exercise to the green-roof detection)
DATA_DIR = Path("../data/preprocessed_articles")

In [ ]:
# Lets check how our input data looks like

text_sources = glob(str(Path(DATA_DIR, "*_cleaned.md")))
print(text_sources)

For now lets load only one file to exemplify the the data mining approach


In [ ]:
if test_mode:
    search_path = glob(str(Path(DATA_DIR, "Koks 2022*_cleaned.md")))
    print("Test mode is ON. Using only a single document for testing.")

## Chunk the text

Split the text into smaller paragraphs, so the LLMs can process them better

We need to encode the text first, to get the number of tokens. Based on this token count the Chunker can measure the text size

In [ ]:
from transformers import AutoTokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat

In [ ]:
# load tokenizer
embed_model = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model),
    max_tokens=256,  # max tokens for MiniLM-l6-v2, set here explicitly for illustration purpose
    # standardize input sizes of chunks for models (padding, truncation)
    padding=True,  # add zero as extra tokens to too short sequences so that they have the same length as other chunks
    truncation=False,  # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
)

## init chunker - based on hierachical chunker but also considers max token length,
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=256,
    # chunk_overlap=0, # --> no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
    split_by_sentence=True,  # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
    merge_peers=True,  # merge smaller chunks, except when at end of paragraph (merge_peers=True),
)


## init Converter that reads in the Markdown documents and converts them to DoclingObjects
md_converter = DocumentConverter(allowed_formats=[InputFormat.MD])

In [ ]:
search_path


## Set up LangGraph: 
AIM: read the text from (various) text sources and extract information about damages of natural hazards. 
In detail, the aim of our data extraction (also called data mining) approach is to extract damages to critical infrastructure (e.g. power blackout due to windstorm, flooded roads, etc.)

What this sub-section is about:
- how to load secrete data (via .env file) 
- chain-of-prompts
- LLM Graph  consisting of two LLM agents 
    - one LLM calling a simple tool
    - the other LLM evaluates the response of the previous LLM (ie. act as LLM-as-a-Judge)
- pydantic Schema  (example of data-classes)


#### LangGraph prompts (chain-of-prompts)


In [ ]:
asset_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Extract every damaged infrastructure asset or service mentioned in the TEXT. The damage has to be caused by a natural hazard.
    Use the entries returned from the "get_infrastructure_entities" tool: decide based on the TEXT for each entry if it really refers to an infrastructure damaged or affected by a natural hazard. If this is the case, you should add the entry to the JSON object.  

    **Important**
    * When an entry contains two or more infrastructure assets then split them in two. For instance: entry: "Road and railway infrastructure" -->  "Road infrastructure", "railway infrastructure"

    ITEM TO EXTRACT:
    - "List of infrastructure assets or services damaged by a natural hazard." 

    """,
        ),
        ("human", "TEXT: {text}"),
    ]
)


location_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """For every infrastructure asset or service returned by ASSETS determine its location by reading the TEXT.
    Store each identified location as an entry in the "location" list.
    DON'T create new items about locations that weren’t there.

    ITEM TO EXTRACT:
    - "location": "List the location of each infrastructure asset or service mentioned in ASSETS."         

    """,
        ),
        (
            "human",
            """
    TEXT: {text}

    ASSETS: {assets}
    """,
        ),
    ]
)


ci_loc_verify_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
    Your task is to act as evaluator, i.e. as an LLM-as-a-Judge. You should recheck the output from a previous LLM which tried to extract damages to critical infrastructure (called ASSET) and their specific locations (called LOCATION).
    Recheck the output by reading the TEXT again. Then, decide for each ASSET-LOCATION-PAIR if it was correctly extracted or not. 
    If the ASSET-LOCATION-PAIR was correctly extracted, return <true>, else <false>.   
    """,
        ),
        (
            "human",
            """
    TEXT: {text}

    ASSET-LOCATION-PAIR: {assets}-{locations}
    """,
        ),
    ]
)

#### Tools - NER


In [ ]:
class NERModel:
    def __init__(self):
        self.nlp_model = self.initialize()

    def initialize(self):
        """Return initialized NER model"""

        dir_model_pipeline = "./spacy_model_pipeline"
        dir_ci_patterns = Path("./ner_patterns.jsonl")

        # LOAD spacy pipeline
        nlp = spacy.load(dir_model_pipeline)

        # add CI_TYPE patterns to spacy nlp model pipeline
        config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
        ## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
        try:
            ruler = nlp.add_pipe("span_ruler", config=config)
            ruler.from_disk(dir_ci_patterns)
        except ValueError:
            print("SpanRuler already exists in pipeline.")
            ruler = nlp.get_pipe("span_ruler")
            ruler.from_disk(dir_ci_patterns)

        # load NER patterns for CI types and their subgroups (needed for cleaning LLm response - STEP 1 before continuing with STEP 2)
        ci_patterns = pd.read_json("ner_patterns.jsonl/patterns", lines=True)

        return nlp


nlp_model = NERModel().nlp_model

In [ ]:
###########################################
# Tools
###########################################

nlp_model = NERModel().nlp_model


@tool
def get_infrastructure_entities(text: str) -> List[str] | None:
    """This tool detects infrastructure assets and services from text by using a spaCy model to conduct Named Entity Recognition (NER)"""

    # remove common misleading CI keywords
    # ("dam" - "damage.*", "train"- "training") as NER patterns not recognize stop-char "\b" for any reason
    # but keep when rest of entity is CI, eg, "damaged rails",
    regex = re.compile(r"\b[Dd]amage\w*|\b[Tt]raining\w*")
    text_clean = re.sub(regex, "", text).replace("  ", " ")

    # get ci entities
    nlp_chunk = nlp_model(text_clean)
    ci_entities = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]

    ## cleaning:
    # remove common errorneous CI keywords ("dam" - "damage.*", "train"- "training") as NER patterns not recognize stop-char "\b" for any reason
    regex = re.compile(r".*damage.*|.*training.*")
    ci_entities = [str(i) for i in ci_entities if not regex.match(str(i))]

    if ci_entities:
        return ci_entities
    else:
        return None

#### LangGraph Graph


In [ ]:
###########################################################################
# LLM OutputSchema
###########################################################################


## LLM allowed Output Schema
class InfrastructureCase(BaseModel):  # OutputSchema of a single damage case
    asset: str
    location: str
    verified: bool


class InfrastructureCases(BaseModel):  # OutputSchema for all damage cases per chunk
    cases: list[InfrastructureCase]


class GraphState(MessagesState):
    text: str
    chunk_relevant: bool
    assets: list[str]
    locations: list[str]
    verified: list[bool]
    cases: InfrastructureCases


###########################################
# Tool  - functional for conditional node
###########################################

## call spacy language model
nlp_model = NERModel().nlp_model

## NOTE: more info for tool calls:
# https://medium.com/@vivekvjnk/introduction-to-tool-use-with-langgraphs-toolnode-0121f3c8c323
# https://docs.langchain.com/oss/python/langchain/tools


def should_continue(state) -> Literal["tools", "locations"]:
    """Route to tool handler, or proceed with info-extraction if already conducted tool called"""

    # Get the last message
    last_message = state["messages"][-1]

    # If the last message is a tool call, check if it's a Done tool call
    if last_message.tool_calls:
        return "tools"
    # Otherwise, proceed with location
    return "locations"


###########################################################################
# LLMs
###########################################################################


# init Agent
llm = ChatNVIDIA(
    model="openai/gpt-oss-20b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_TOKEN"),
    temperature=0.01,
    seed=42,
)

## init SpecificAgent
llm_verify = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_TOKEN"),
    temperature=0.01,
    seed=42,
)


####################################
# Bind Tools to 1. LLM
####################################


# bind tool to graph-node(s) of 1st model
tools = [get_infrastructure_entities]

tool_node = ToolNode(tools)

llm_tool = llm.bind_tools(tools, tool_choice="any", parallel_tool_calls=False)


###########################################################################
# Nodes
###########################################################################


def extract_assets(state):
    messages = state["messages"]
    if len(messages) == 0:
        messages = [HumanMessage(content=asset_prompt.format(text=state["text"]))]
    result = llm_tool.invoke(messages)
    print("asset:", result)

    i = 0
    while not result.tool_calls or len(result.tool_calls) == 0:
        print(f"⚠️ No tool calls were returned by the model.Trying again [{i}]")
        result = llm_tool.invoke(messages)
        if i == 5:
            return None
        i += 1

    else:
        print("! tool called:", result.tool_calls)

    # NOTE ToolNode expects a list of messages
    return {"messages": [result]}


# process tool call for next node
def asset_finalize(state):
    result = state["messages"][-1].content
    print("Asset finalize", len(ast.literal_eval(result)), result)

    result = ast.literal_eval(result)  # make str[list[str]] -> list[str]
    return {"assets": result}


def extract_locations(state) -> GraphState:
    # ensure that for each infrastructure asset - a location is extracted
    class Location(BaseModel):
        location: Annotated[
            List[str],
            Len(min_length=len(state["assets"]), max_length=len(state["assets"])),
        ]
        # longitude:
        # latitude:

    # bind PydanticSchema to LLM
    llm_structured = llm.with_structured_output(Location)

    result = llm_structured.invoke(
        [
            HumanMessage(
                content=location_prompt.format(
                    text=state["text"],
                    assets=state["assets"],
                )
            )
        ]
    )

    # try again if could not extract location (None) or location info is shorter than no. of assets
    # TODO force LLM with LangGraph func/method to output response in certain length
    i = 0
    while (result is None) or (len(result.location) < len(state["assets"])):
        result = llm_structured.invoke(
            [
                HumanMessage(
                    content=location_prompt.format(
                        text=state["text"], assets=state["assets"]
                    )
                )
            ]
        )

        # when extraction takes too long
        # --> extract locations for each single asset

        class SingleLocation(BaseModel):
            location: Annotated[list[str], Len(min_length=1, max_length=1)] | str = (
                "NAN"
            )

        if i == 5:
            assets_list = state["assets"]
            location_list = []

            # bind PydanticSchema to LLM
            llm_structured = llm.with_structured_output(SingleLocation)

            for asset in assets_list:
                result = llm_structured.invoke(
                    [
                        HumanMessage(
                            content=location_prompt.format(
                                text=state["text"], assets=asset
                            )
                        )
                    ]
                )
                try:
                    location_list.append(result.location)
                except AttributeError:
                    location_list.append("NAN")
            print("locations_list", location_list)
            return {"locations": location_list}
        i += 1

    return {"locations": result.location}


def verify(state):
    # define Output Schema
    llm_structured = llm_verify.with_structured_output(InfrastructureCase)

    result = llm_structured.invoke(
        [
            HumanMessage(
                content=ci_loc_verify_prompt.format(
                    text=state["text"],
                    assets=state["assets"],
                    locations=state["locations"],
                )
            )
        ]
    )
    print(
        "veri",
        HumanMessage(
            content=ci_loc_verify_prompt.format(
                text=state["text"],
                assets=state["assets"],
                locations=state["locations"],
            )
        ),
    )
    print("verify", result)

    return {
        "verified": result,
    }


## init  Graph
builder = StateGraph(GraphState)


builder.add_node(
    "assets", extract_assets
)  # NOTE structured output need to be called after tool binding, not together
builder.add_node("tools", tool_node)
builder.add_node("asset_finalize", asset_finalize)
builder.add_node("locations", extract_locations)
builder.add_node("verify", verify)


## define arcs (ie. graph edges)
builder.set_entry_point("assets")
builder.add_conditional_edges("assets", should_continue, ["tools", "asset_finalize"])
builder.add_edge("tools", "asset_finalize")
builder.add_edge("asset_finalize", "locations")
builder.add_edge("locations", "verify")
builder.add_edge("verify", END)


# init graph
graph = builder.compile()
graph

# # for exporting graph
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))

## Conduct Data mining

In [ ]:
## store the response of our LLM agents here
df_results = pd.DataFrame()

import time

## Start CI impact extraction
for file_no, filename in enumerate(search_path):
    # Load the existing cleaned markdown file
    doclingdoc = md_converter.convert(filename).document

    print("Chunking document...")
    chunk_iter = chunker.chunk(dl_doc=doclingdoc)
    doc = list(chunk_iter)

    # scalene_profile.start()  # NOTE - try out scalene (or any other Python Profiler) if you want to find bottlenecks in the code where computation takes very long

    # iterate over chunks and extract CI impacts
    for chunk_no, chunk in enumerate(doc):
        try:
            result = graph.invoke(
                {
                    "text": chunk.text,
                }
            )
            print(result)

            # NOTE: if you get a ResourceExhaust message - Exception[503]
            # i.e. the NVIDIA API cant be reached -> catch the error and let the reuqest stream sleep for a minute
        except Exception as e:
            print(f"! ERROR {e}. Sleeping for 1 min and then try again")
            time.sleep(60)
            try:
                result = graph.invoke(
                    {
                        "text": chunk.text,
                    }
                )
                print(result)
            except ValueError as e:
                print(f"! ERROR {e}. Erroneous LLM response. Going to next chunk")
                continue

        # post process response and store it to df_results
        df_result = pd.DataFrame(result["verified"].model_dump(), index=[0])
        df_result["chunk_text"] = chunk.text
        try:
            df_results = pd.concat([df_results, df_result])
        except Exception as e:
            print(
                f"Could not add LLm response to final output dataframe\nError M:{e}\nErrroneous response: {df_result}"
            )
            continue

### Have a look at the extracted infrastructure damages 

Task:
* Does the extracted cases of damaged infrastructure and their location sense? Does one of the LLMs potentially hallucinate a lot?

In [ ]:
df_results

# Excurse: Tests in pytest and unittest

In [ ]:
# standard library
import os
import requests
import urllib

# NOTE: in this example we use actually pytest for testing our functions,
# but the nice thing is that pytest can be enriched with objects coming from unittest, such as "patches", mocked uniitest-objects
import unittest
from unittest.mock import patch

# third party imports
import requests_mock
import pandas as pd

In [ ]:
def add(a, b):
    return a + b

In [ ]:
# pytest in jupyter #1
# #1 minimum version, but less informative


def test_add():
    assert add(2, 2) == 4
    print("passed")


test_add()

In [ ]:
# pytest in jupyter #2
# #2 better pytest usage

# NOTE: first run in terminal or here: !uv add ipytest
import ipytest

ipytest.autoconfig()  # load configuration of ipytest

In [ ]:
%%ipytest
# 1. Alternative (pytest): run test with %%ipytest (a line magic function, similar as `%%shell` for executing shell code in jupyter cell) for each cell with a test <-- runs only tests in the cell


def test_add():
    assert add(2, 2) == 4

In [ ]:
# 2. Alternative (pytest): put a `ipytest.run()` in a cell after oyour tests <-- runs all tests coming from different cells


def test_add():
    assert add(2, 2) == 4
    print("passed")

In [ ]:
def test_add_v2():
    assert add(3, 1) == 5

In [ ]:
ipytest.run()

In [ ]:
# unittest in jupyter


class TestAdd(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)


unittest.main(
    argv=[""], verbosity=2, exit=False
)  # argv: pass command line arguments to unittest executer, verbosity: how much info should be printed out, exit: prevents  unitest-main() to exit the program after running tests


## unittest are executed in python scripts (eg. in a test_xx.py) a bit differently than in jb notebooks:
# Example: for a `if __name__ == "__main__": `-block, here in regard to unittest checks
# if __name__ == '__main__':
#     unittest.main()

# NOTE:
# The `if __name__ == "__main__": `-block makes sure that the code inside this block only runs when the file is directly executed.
# For instance, when another file imports this file as module, the code is not being executed.

## Mocking

**How to write a test with a mock?**\

**NOTE:** "Mocking is the solution when your unit test would call a service that’s outside of your domain." [Source: https://www.signadot.com/blog/why-developers-shouldnt-write-mocks-a-guide-to-modern-testing/]
Your mocked function (or mocked code in general) acts in a way that it simulates another service (e.g., an API) or some data, etc.
In other words, we use mocking when we want to replace a function/data etc. with a dummy representation of it.
For example, if we need an API call for a test function, then we dont want to call the API each time when we run the test.
For such a case, we mock the API call and provide a Dummy version of the API call 


In [ ]:
## as a function


def nominatim_query(location: str) -> dict:
    base_url = "https://nominatim.openstreetmap.org/"
    session = requests.Session()
    cache = {}

    # enrich nominatim url with location info
    # formatted_query = urllib.parse.quote(location)
    url_metainfo = base_url + f"search?q={location}&format=geocodejson"
    url_metainfo += (
        "&polygon_geojson=1&addressdetails=1&namedetails=1&accept-language=en"
    )

    headers = {"User-agent": None}

    # Get nominatim output
    r_metainfo = session.get(
        url_metainfo, timeout=20, headers=headers
    )  # Adjust the timeout as needed

    if r_metainfo.status_code != 200:
        print(f"Unexpected status code: {r_metainfo.status_code}")

    r_metainfo_json = r_metainfo.json()
    cache.update({"query": r_metainfo_json})

    return r_metainfo_json


location = "Saxony"
rr = nominatim_query(location)

rr["features"][0]["properties"]

## For good practice: lets create a small class

This class should take our function nominatim_query() as method

In [ ]:
class Gazetteer:
    def __init__(
        self, gazetteer_source: str = "nominatm"
    ):  # <-- initalize the objects/classes attributes
        self.gazetteer_source = gazetteer_source  # NOTE: <- object attribute (here it is a attirbute that stores our defined value "nominatim" from below)
        self.base_url = "https://nominatim.openstreetmap.org/"
        self.session = requests.Session()
        self.cache = {}

    def nominatim_query(
        self, location: str
    ) -> dict:  #  <-- self: refers to the current object
        # enrich nominatim url with location info and request also meta-info about location (e.g. coordinates, location is passed in english, etc.)
        location = urllib.parse.quote(
            location
        )  # encode a string (with special characters, like ä, à, ß) into a URL-safe representation
        url_metainfo = self.base_url + f"search?q={location}&format=geocodejson"
        url_metainfo += (
            "&polygon_geojson=1&addressdetails=1&namedetails=1&accept-language=en"
        )

        # Get nominatim output
        r_metainfo = self.session.get(
            url_metainfo, timeout=20, headers={"User-agent": None}
        )

        # sanity check
        # NOTE: The syntax is assert <conditional>, <optional message>. Optional: a message is printed if the assertion fails
        # Example: check if the API request was successful
        assert (
            r_metainfo.status_code == 200
        ), f"Unexpected status code: {r_metainfo.status_code}"

        # Alternative to sanity check:
        # if r_metainfo.status_code != 200:
        #     print(f"Unexpected status code: {r_metainfo.status_code}")

        # convert API response -> json, update cache
        r_metainfo_json = r_metainfo.json()
        self.cache.update({"query": r_metainfo_json})

        return r_metainfo_json


location = "Saxony"
rr = Gazetteer(gazetteer_source="nominatim").nominatim_query(location)
# NOTE: setting explicitly a value for the object parameter "gazetteer_source" is not needed,
# when it is identical to the default defined in the class object (see, above)

rr.keys()
rr["features"][0]["properties"]

In [ ]:
## SIDE-NOTE:
# classes that depend on each other (see, https://www.geeksforgeeks.org/python/__init__-in-python/)

## Minimum Example:
# class A:
#     def __init__(self):
#         print("A init called")

# class B(A):
#     def __init__(self):
#         super().__init__()   # <--- Call parent __init__
#         print("B init called")

# obj = B()


# Explanation:
#     When obj = B() is created, Python first calls B.__init__(). Inside B.__init__, the line super().__init__() calls the parent’s (A) constructor.
#     As a result, "A init called" is printed first.
#     After the parent class initialization completes, remaining code in B.__init__() executes and "B init called" is printed.

### lets test the method inside this class

In [ ]:
%%ipytest


@requests_mock.mock()
def mock_request(m):
    m.get(url, text="success")
    return requests.get(url).text


class TestGazetteer(unittest.TestCase):
    def setUp(self):
        self.nom_gaz = Gazetteer(gazetteer_source="nominatim")

    @requests_mock.mock()
    def test_nominatim_query_calls(self, m):
        gaz = Gazetteer(gazetteer_source="nominatim")

        # set up mock for nominatim api
        # take any location in OSM as dummy that exists in really,eg. Saxony, the only need is to check if nominatim_query() works
        url = "https://nominatim.openstreetmap.org/search?q=Saxony&format=geocodejson&polygon_geojson=1&addressdetails=1&namedetails=1&accept-language=en"
        json_out = '[{"name":"test"}]'
        m.get(url, text=json_out)
        print(m.get(url, text=json_out))
        with patch.object(gaz, "nominatim_query", wraps=gaz.nominatim_query) as mock_nom_query:
            _ = gaz.nominatim_query(
                "Saxony"
            )  # test mocked nominatim_query() with our Dummy Location "Saxony"; it should give same response as for variable "url"
            mock_nom_query.assert_called()

# Voila - we are done with the text-2-data exercise and some texting (with mocks)